# Data Visualization Practice: Audience-Aware Techniques — SOLUTION

**Complete implementations with explanations, alternate approaches, extended practice, and simulation guidance.**

This notebook mirrors the Skeleton but provides full working code, multiple ways to achieve the same result, detailed commentary on *why* certain choices help different audiences, and richer simulation experiments.

Use it to check your work, learn alternate patterns, and deepen your understanding of how these techniques serve real data analysis report needs.

## Flowchart: Audience-Aware Visualization Workflow

```mermaid
flowchart TD
    Start[Start: Define Purpose & Primary Audience] --> Audience[Analyze Audience]
    Audience --> DL{Data Literacy Level?}
    DL -->|High (Experts/Technicians)| Advanced[Advanced Viz: Error bars, Normalized hists, Detailed stats]
    DL -->|Low (Executives/Nonspecialists)| Simple[Simple Viz: Basic bars, Pies, Clear labels, Minimal jargon]
    Audience --> SK{Subject Knowledge?}
    SK -->|High| SkipIntro[Skip basic explanations, use domain conventions]
    SK -->|Low| Explain[Provide context, highlight insights, explain 'good/bad']
    Audience --> TS{Time Span Available?}
    TS -->|Short (C-level, skimmers)| Quick[Large fonts, 1 key insight per viz, annotations]
    TS -->|Long| Detail[More layers, multiple related viz, appendix details]
    Simple & Advanced --> Implement[Implement with Matplotlib / Pandas / Seaborn]
    SkipIntro & Explain --> Implement
    Quick & Detail --> Implement
    Implement --> Validate[Validate: Does this help THIS audience reach the insight quickly & accurately?]
    Validate --> Report[Place in Data Analysis Report: Body for main points, Appendix for technical depth]
    Report --> End[Deliver clear, audience-resonant message]
```


## Setup: Imports and Sample Datasets

Same reproducible datasets as the skeleton. We print summaries so you can see the numbers behind the pictures.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import FancyBboxPatch  # for nice annotations if wanted

sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)

regions = ['North America', 'Europe', 'Asia Pacific', 'Latin America']
sales_q1 = np.array([245.0, 312.0, 189.0, 98.0])
sales_q2 = np.array([278.0, 295.0, 210.0, 115.0])
errors_q1 = np.array([18.0, 25.0, 15.0, 9.0])
errors_q2 = np.array([20.0, 22.0, 17.0, 11.0])

df_sales = pd.DataFrame({
    'Region': regions,
    'Q1_Sales': sales_q1,
    'Q2_Sales': sales_q2,
    'Q1_Error': errors_q1,
    'Q2_Error': errors_q2
})
print('=== Regional Sales Data ===')
print(df_sales.to_string(index=False))

months = np.arange(1, 13)
base_trend = 50 + 8 * np.sin(months / 1.8)
engagement = base_trend + np.random.normal(0, 4, 12)
engagement_err = np.abs(np.random.normal(3.5, 1.2, 12))

print('\n=== Engagement Time Series (sample) ===')
print(pd.DataFrame({'Month': months[:6], 'Engagement': np.round(engagement[:6],1), '±Error': np.round(engagement_err[:6],1)}).to_string(index=False))

content_types = ['Technical Reports', 'Interactive Viz', 'Tutorials', 'Case Studies']
audience_reach = np.array([28, 35, 22, 15])

novice_scores = np.random.normal(loc=62, scale=14, size=180)
expert_scores = np.random.normal(loc=81, scale=7, size=140)

print('\n=== Score Distributions Ready ===')
print(f'Novice (n={len(novice_scores)}): mean={novice_scores.mean():.1f}, std={novice_scores.std():.1f}')
print(f'Expert  (n={len(expert_scores)}): mean={expert_scores.mean():.1f}, std={expert_scores.std():.1f}')


## 1. Comparing Categories with Bar Graphs — Complete Solutions

Three different approaches shown: pure Matplotlib (most control), Pandas `.plot()` (convenient for DataFrames), and Seaborn (beautiful defaults + statistical layers).

**Audience tip**: For executives, add large value labels and a clear takeaway sentence in the title or caption.

In [ ]:
# SOLUTION 1.1: Vertical bar chart - Pure Matplotlib + value labels
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(df_sales['Region'], df_sales['Q1_Sales'], 
              color='#2E86AB', edgecolor='black', linewidth=0.8)

ax.set_title('Q1 Sales by Region — Executive Snapshot', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Region', fontsize=11)
ax.set_ylabel('Sales (thousands USD)', fontsize=11)
plt.xticks(rotation=15, ha='right')

# Add value labels on bars (very helpful for quick reading)
ax.bar_label(bars, fmt='%.0f', padding=4, fontsize=10, fontweight='medium')

# Add subtle mean reference line
mean_val = df_sales['Q1_Sales'].mean()
ax.axhline(mean_val, color='gray', linestyle='--', linewidth=1.2, alpha=0.7, label=f'Mean = {mean_val:.0f}')
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
# SOLUTION 1.2: Horizontal bar chart - Pandas plot (very concise)
fig, ax = plt.subplots(figsize=(8, 5))

df_sales.sort_values('Q1_Sales', ascending=True).plot(
    kind='barh', x='Region', y='Q1_Sales', ax=ax,
    color='#A23B72', legend=False, edgecolor='black', linewidth=0.7
)

ax.set_title('Q1 Sales by Region (Horizontal — easier label reading)', fontsize=13, fontweight='bold')
ax.set_xlabel('Sales (thousands USD)')
ax.set_ylabel('')

# Value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=4, fontsize=10)

plt.tight_layout()
plt.show()


**Alternate with Seaborn** (recommended for most exploratory work — cleaner aesthetics and easy `hue` later):

In [ ]:
# SOLUTION 1.3: Seaborn barplot (excellent defaults, easy extensions)
fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(data=df_sales, x='Region', y='Q1_Sales', ax=ax, 
            color='#F18F01', edgecolor='black', linewidth=0.8)

ax.set_title('Q1 Sales by Region — Seaborn Style (clean & modern)', fontsize=13, fontweight='bold')
ax.set_ylabel('Sales (thousands USD)')
plt.xticks(rotation=15, ha='right')

# Quick value labels
for p in ax.patches:
    ax.annotate(f'{p.get_height():.0f}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=10, fontweight='medium')

plt.tight_layout()
plt.show()


## 2. Adding Error Bars — Complete

Error bars are one of the most important additions for **technical credibility**. They prevent over-interpretation of small differences.

In [ ]:
# SOLUTION 2.1: Bar + Error bars (Matplotlib)
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(df_sales['Region'], df_sales['Q1_Sales'], 
              yerr=df_sales['Q1_Error'], capsize=6,
              color='#264653', edgecolor='black', alpha=0.85,
              error_kw={'linewidth': 2, 'ecolor': '#E76F51'})

ax.set_title('Q1 Sales by Region with Uncertainty (Error Bars)\nTechnical audiences can assess whether differences are reliable', 
             fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel('Sales (thousands USD)')
plt.xticks(rotation=15, ha='right')

ax.bar_label(bars, fmt='%.0f', padding=8, fontsize=9)

plt.tight_layout()
plt.show()


## 3. Shaded Error with `fill_between` — Complete

The shaded band gives an immediate visual sense of variability around the trend. Very effective for time-series insights in reports.

In [ ]:
# SOLUTION 3.1: Line + fill_between shaded uncertainty
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(months, engagement, marker='o', markersize=5, linewidth=2.2,
        color='#2A9D8F', label='Observed Engagement')

lower = engagement - engagement_err
upper = engagement + engagement_err
ax.fill_between(months, lower, upper, alpha=0.28, color='#2A9D8F',
                label='Uncertainty band (±1σ)')

ax.set_title('Monthly Engagement Trend with Uncertainty Band\n(Shaded region shows variability — useful for analysts & technical readers)', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Engagement Score')
ax.legend(loc='upper right', framealpha=0.9)
ax.set_xticks(months)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Stacked Bar Graphs — Complete

Pandas `stacked=True` is the easiest route. Seaborn does not have a direct stacked barplot; we usually fall back to pandas or manual matplotlib for stacks.

In [ ]:
# SOLUTION 4.1: Stacked bar (Pandas) — shows total + composition
fig, ax = plt.subplots(figsize=(9, 5))

df_sales.plot(kind='bar', x='Region', y=['Q1_Sales', 'Q2_Sales'],
              stacked=True, ax=ax, color=['#264653', '#E9C46A'],
              edgecolor='black', linewidth=0.6)

ax.set_title('Total Sales by Region — Q1 + Q2 Composition (Stacked)\nGood for seeing both total performance and quarterly breakdown', 
             fontsize=12, fontweight='bold')
ax.set_ylabel('Total Sales (thousands USD)')
ax.set_xlabel('Region')
plt.xticks(rotation=15, ha='right')
ax.legend(title='Quarter', loc='upper right')

plt.tight_layout()
plt.show()


## 5. Side-by-Side (Grouped) Bars — Complete

Seaborn's `hue` parameter makes grouped bars trivial and beautiful.

In [ ]:
# SOLUTION 5.1: Grouped bars with Seaborn (best for comparison within categories)
# First melt the dataframe for long format (Seaborn prefers long)
df_long = df_sales.melt(id_vars=['Region'], value_vars=['Q1_Sales', 'Q2_Sales'],
                        var_name='Quarter', value_name='Sales')

fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(data=df_long, x='Region', y='Sales', hue='Quarter', ax=ax,
            palette=['#2E86AB', '#F18F01'], edgecolor='black', linewidth=0.7)

ax.set_title('Q1 vs Q2 Sales by Region (Grouped / Side-by-Side)\nEasiest format for direct quarter-to-quarter comparison per region', 
             fontsize=12, fontweight='bold')
ax.set_ylabel('Sales (thousands USD)')
plt.xticks(rotation=15, ha='right')
ax.legend(title='Quarter')

plt.tight_layout()
plt.show()


## 6. Pie Charts — Complete & Formatted

**Important audience note**: Pies work best with ≤5 slices and when the audience cares about 'share of total'. For precise ranking or many categories, bars are usually superior.

In [ ]:
# SOLUTION 6.1: Professional pie chart with explode, custom colors, legend
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#264653', '#2A9D8F', '#E9C46A', '#F4A261']
explode = (0.0, 0.10, 0.0, 0.0)  # Pull out the largest slice

wedges, texts, autotexts = ax.pie(
    audience_reach, labels=content_types, autopct='%.1f%%',
    startangle=90, explode=explode, colors=colors,
    pctdistance=0.72, labeldistance=1.15,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)

# Style percentage text
for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight('bold')
    autotext.set_color('white')

ax.set_title('Audience Reach by Content Type\n(Interactive Viz dominates — insight for content strategy team)', 
             fontsize=13, fontweight='bold', pad=15)

ax.legend(wedges, content_types, title='Content Type', loc='center left',
          bbox_to_anchor=(1, 0, 0.5, 1), fontsize=10)

plt.tight_layout()
plt.show()


## 7 & 8. Histograms (Raw + Normalized) — Complete

We show both count (frequency) and density (normalized) versions side-by-side so you can see the perceptual difference. Normalized is almost always preferable when comparing groups of different sizes.

In [ ]:
# SOLUTION 7+8: Side-by-side raw count vs normalized density histograms
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# LEFT: Raw counts (misleading if n different)
axes[0].hist(novice_scores, bins=18, color='#E76F51', alpha=0.65, edgecolor='white', label='Novice (n=180)')
axes[0].hist(expert_scores, bins=18, color='#2A9D8F', alpha=0.65, edgecolor='white', label='Expert (n=140)')
axes[0].set_title('Raw Count Histograms\n(Larger group visually dominates)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Comprehension Score')
axes[0].set_ylabel('Count (Frequency)')
axes[0].legend()

# RIGHT: Normalized density (fair shape comparison)
axes[1].hist(novice_scores, bins=18, density=True, color='#E76F51', alpha=0.65, edgecolor='white', label='Novice')
axes[1].hist(expert_scores, bins=18, density=True, color='#2A9D8F', alpha=0.65, edgecolor='white', label='Expert')
axes[1].set_title('Normalized (Density) Histograms\n(Shape comparison — recommended for unequal n)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Comprehension Score')
axes[1].set_ylabel('Density (area under curve = 1)')
axes[1].legend()

fig.suptitle('Audience Comprehension Score Distributions — Impact of Normalization', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Seaborn alternate** for even nicer histograms with automatic KDE:

In [ ]:
# SOLUTION 8 alt: Seaborn histplot with density + KDE (very polished)
fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(novice_scores, bins=20, stat='density', color='#E76F51', alpha=0.5, 
             kde=True, label='Novice', ax=ax)
sns.histplot(expert_scores, bins=20, stat='density', color='#2A9D8F', alpha=0.5,
             kde=True, label='Expert', ax=ax)

ax.set_title('Seaborn: Normalized Histograms + KDE Curves\n(Easy, beautiful, statistically aware)', fontsize=12, fontweight='bold')
ax.set_xlabel('Comprehension Score')
ax.set_ylabel('Density')
ax.legend(title='Audience Group')

plt.tight_layout()
plt.show()


## Extended Practice & Simulation Section

This section goes beyond the skeleton. Experiment freely — the goal is to build visual intuition and understand audience impact.

In [ ]:
# EXTENDED SIMULATION: Change parameters and re-run any plot above
# Try these scenarios one by one:

# SCENARIO A: Executive-friendly simplification
# Reduce to top 3 regions only, remove error bars, add big takeaway annotation

# SCENARIO B: Technical deep-dive
# Add 95% CI error bars (multiply errors by ~1.96 if they were SE), overlay individual data points with stripplot

# SCENARIO C: Distribution shift
# Make expert_scores much closer to novice (change loc to 68) — how does the normalized histogram story change?

# SCENARIO D: 100% Stacked version (composition only)
df_sales['Total'] = df_sales['Q1_Sales'] + df_sales['Q2_Sales']
df_sales['Q1_pct'] = df_sales['Q1_Sales'] / df_sales['Total'] * 100
df_sales['Q2_pct'] = df_sales['Q2_Sales'] / df_sales['Total'] * 100

print('Added percentage columns for 100% stacked simulation.')
print(df_sales[['Region', 'Q1_pct', 'Q2_pct']].round(1))


In [ ]:
# BONUS: 100% Stacked bar (composition-focused, good for some executive views)
fig, ax = plt.subplots(figsize=(9, 5))

df_pct = df_sales[['Region', 'Q1_pct', 'Q2_pct']].set_index('Region')
df_pct.plot(kind='bar', stacked=True, ax=ax, color=['#264653', '#E9C46A'],
            edgecolor='white', linewidth=1)

ax.set_title('100% Stacked: Quarterly Composition of Total Sales per Region\n(Shows mix, not absolute performance — use when totals are less important)', 
             fontsize=11, fontweight='bold')
ax.set_ylabel('Percentage of Region\'s Total Sales')
ax.set_ylim(0, 100)
plt.xticks(rotation=15, ha='right')
ax.legend(title='Quarter', loc='upper right')

# Add percentage labels inside bars (advanced but very useful)
for c in ax.containers:
    ax.bar_label(c, fmt='%.0f%%', label_type='center', fontsize=9, color='white', fontweight='bold')

plt.tight_layout()
plt.show()


## Reflection: Connecting Techniques to Audience & Report Structure

From the provided documents:

- **Primary audience (collaborator/client)**: They want the story fast. Use clear grouped or stacked bars with annotations in the Body of your data analysis report.
- **Executive secondary audience**: Skim. They need the headline insight in < 5 seconds. Big titles, large labels, minimal ink (Tufte), one clear visual per question.
- **Technical supervisor**: They check quality. Error bars, normalized distributions, and reproducible code in the Appendix give them confidence in your work.

**Best practice checklist** (from audience analysis principles):
1. Identify the 1-2 key questions your audience has.
2. Choose the simplest visualization that answers those questions accurately.
3. Add exactly the detail needed (error bars? only if they change interpretation).
4. Test mentally: Can a tired executive get the point in 3 seconds? Can an analyst verify the numbers?
5. Label everything clearly — never assume the reader knows your color coding.

You now have a powerful toolkit. Practice adapting these visualizations to different hypothetical audiences and you will produce data analysis reports that are both beautiful and effective.

**End of Solution Notebook** — Happy visualizing!